# Notebook 01 — Limpieza de Datos y Construcción de Variables

Este notebook prepara el dataset modelable a partir del CSV bruto de Kaggle, aplicando las reglas metodológicas que evitan fugas de información y garantizan la reproducibilidad:

- **Verificación obligatoria** de la codificación de `Showed_up` (1 = asistió, 0 = no-show) antes de cualquier modelado.
- **Codificación por frecuencia** de `Neighbourhood`, ajustada únicamente sobre el conjunto de entrenamiento (NUNCA codificación basada en outcome).
- **Regla estricta de tiempo de decisión** para el historial del paciente: para cada cita, sólo se usan citas previas cuyo `AppointmentDay < ScheduledDay` actual; la propia fila NUNCA cuenta como su propio antecedente (incluso en filas anómalas con `AppointmentDay < ScheduledDay`).
- **División temporal** por percentil 80 de `AppointmentDay` (no aleatoria, dado que los pacientes se repiten).
- **Semillas fijas** y nombres versionados de los ficheros de salida.

La lógica pesada vive en `src/preparacion.py`; este notebook orquesta y narra. Salidas finales:

- `datos/procesados/full_clean_v1.csv` — dataset completo para el IPW (NB04).
- `datos/procesados/train_v1.csv`, `datos/procesados/test_v1.csv` — split temporal para el modelo predictivo (NB03).
- `outputs/reportes/nb01_metadatos_v1.json` — sidecar de números metodológicos clave (defensibilidad).

In [1]:
# Imports y configuración global
from __future__ import annotations

import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Permitir importar el paquete src/ desde el notebook
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import preparacion as prep
from src.rutas import (
    CSV_RAW,
    CSV_FULL_CLEAN,
    CSV_TRAIN,
    CSV_TEST,
    DATOS_PROCESADOS,
    REPORTES,
    SEED,
)

# Logging hacia stdout para que las cabeceras informativas de cada paso
# aparezcan en las celdas del notebook.
logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    stream=sys.stdout,
    force=True,
)

# Semilla global para reproducibilidad. Cualquier paso estocástico aguas
# abajo debe partir de esta semilla.
np.random.seed(SEED)

DATOS_PROCESADOS.mkdir(parents=True, exist_ok=True)
REPORTES.mkdir(parents=True, exist_ok=True)
print(f"Semilla fijada en {SEED}. Ruta de salida: {DATOS_PROCESADOS}")

Semilla fijada en 42. Ruta de salida: /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/datos/procesados


## 1. Carga del dataset bruto

Cargamos el CSV original sin transformaciones para inspeccionar dtypes, nulos y duplicados antes de modificar nada.

In [2]:
df = prep.cargar_datos_brutos(CSV_RAW)
df_bruto = df.copy()  # snapshot pre-limpieza para los metadatos defensivos

print(f"Forma: {df.shape}")
print()
print("Dtypes por columna:")
print(df.dtypes)
print()
print("Nulos por columna:")
print(df.isna().sum())
print()
print(f"Duplicados completos: {df.duplicated().sum()}")
print(f"AppointmentID duplicados: {df['AppointmentID'].duplicated().sum()}")

Cargado dataset bruto: 106987 filas, 15 columnas


Forma: (106987, 15)

Dtypes por columna:
PatientId         float64
AppointmentID       int64
Gender                str
ScheduledDay          str
AppointmentDay        str
Age                 int64
Neighbourhood         str
Scholarship          bool
Hipertension         bool
Diabetes             bool
Alcoholism           bool
Handcap              bool
SMS_received         bool
Showed_up            bool
Date.diff           int64
dtype: object

Nulos por columna:
PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
Showed_up         0
Date.diff         0
dtype: int64

Duplicados completos: 0
AppointmentID duplicados: 0


## 2. Verificación de la codificación de `Showed_up`

**Convención obligatoria** para todo el TFG: `Showed_up = 1` significa que el paciente asistió, `Showed_up = 0` significa no-show. Una codificación invertida que sobreviva hasta la defensa invertiría todos los signos del ATE, las direcciones de SHAP y las fórmulas del Monte Carlo.

`prep.verificar_codificacion_showed_up` realiza tres comprobaciones: distribución por clase, exigencia de tasa de asistencia ~80% (firma del dataset), y conversión final a entero 0/1.

In [3]:
df = prep.verificar_codificacion_showed_up(df)
print()
print(f"Tipo final de Showed_up: {df['Showed_up'].dtype}")
print(f"Distribución: {df['Showed_up'].value_counts().to_dict()}")

Showed_up: 85307 asistieron (79.7%), 21680 no-shows (20.3%)



Tipo final de Showed_up: int64
Distribución: {1: 85307, 0: 21680}


## 3. Limpieza de valores imposibles

- Convertimos `PatientId` a entero (en el CSV viene como `float64` por la magnitud de los IDs); evita sorpresas de igualdad en los `groupby` aguas abajo, sobre todo en el bootstrap por clúster del NB04.
- Eliminamos filas con `Age < 0` (físicamente imposibles).
- Documentamos —**no eliminamos**— las filas con `AppointmentDay < ScheduledDay`. Esa anomalía puede reflejar reagendamientos o errores de tipeo; eliminarla introduciría sesgo no controlado.
- Inspeccionamos `Handcap`: el plan menciona la posibilidad de codificación multinivel inconsistente, pero en este CSV es booleano; lo documentamos y no lo recodificamos.

In [4]:
df = prep.limpiar_valores_imposibles(df)

Eliminadas 0 filas con Age < 0


AppointmentDay < ScheduledDay (anomalía documentada, no eliminada): 5 filas (0.00%)


Valores únicos de Handcap: [False, True] (binario, no requiere recodificación)


Limpieza: 106987 → 106987 filas (0 eliminadas)


## 4. Parseo de fechas y verificación de `Date.diff`

Convertimos `ScheduledDay` y `AppointmentDay` a `datetime`. Recalculamos `Date.diff` en días y comparamos con la columna existente; el recálculo es autoritativo.

In [5]:
df = prep.parsear_fechas(df)
print()
print("Rango de ScheduledDay: ", df['ScheduledDay'].min().date(), '→', df['ScheduledDay'].max().date())
print("Rango de AppointmentDay:", df['AppointmentDay'].min().date(), '→', df['AppointmentDay'].max().date())
print()
print("Lead time (días) — describe:")
print(df['Date.diff'].describe())

Date.diff: 0 inconsistencias entre la columna existente y el recálculo (se sobrescribe con el recálculo, autoritativo)



Rango de ScheduledDay:  2015-11-10 → 2016-06-08
Rango de AppointmentDay: 2016-04-29 → 2016-06-08

Lead time (días) — describe:
count    106987.000000
mean         10.166721
std          15.263508
min          -6.000000
25%           0.000000
50%           4.000000
75%          14.000000
max         179.000000
Name: Date.diff, dtype: float64


## 5. Construcción de variables derivadas (sin historial)

Variables que dependen sólo de la fila actual:

| Variable | Construcción |
|---|---|
| `lead_time` | `Date.diff` (días entre programación y cita) |
| `lead_time_bin` | bins operativos: `mismo_dia`, `1-7d`, `8-14d`, `15+d` |
| `age_band` | bins clínicos: `0-18`, `19-35`, `36-55`, `56-70`, `70+` |
| `comorbidity_count` | suma de Hipertension + Diabetes + Alcoholism + Handcap |
| `chronic_flag` | 1 si comorbidity_count > 0 |
| `scheduled_*` | día de la semana / mes del momento de programación (la hora se elimina si es constante) |
| `appointment_*` | día de la semana / mes del momento de la cita |

Ni `Showed_up` ni `SMS_received` entran aquí: el outcome y el tratamiento se mantienen completamente separados de la ingeniería de variables.

In [6]:
df = prep.crear_features_basicas(df)
print()
print("age_band:")
print(df['age_band'].value_counts().sort_index())
print()
print("lead_time_bin:")
print(df['lead_time_bin'].value_counts())
print()
print("comorbidity_count:")
print(df['comorbidity_count'].value_counts().sort_index())

scheduled_hour es constante (todo 0) — eliminada de las salidas



age_band:
age_band
0-18     25327
19-35    24137
36-55    30019
56-70    18931
70+       8573
Name: count, dtype: int64

lead_time_bin:
lead_time_bin
mismo_dia    37159
1-7d         31446
15+d         26738
8-14d        11644
Name: count, dtype: int64

comorbidity_count:
comorbidity_count
0    80576
1    18122
2     7658
3      618
4       13
Name: count, dtype: int64


## 6. Historial del paciente — regla estricta de tiempo de decisión

Este es el paso de feature engineering con mayor riesgo de fuga.

**Regla estricta:** para cada cita del paciente $p$ con `ScheduledDay = S`, sólo cuentan como historial las citas previas del mismo paciente cuyo `AppointmentDay < S`. Es decir: el resultado de esas citas **ya se conocía** en el momento en que se tomó la decisión sobre la cita actual.

Esta regla es estrictamente más fuerte que un *expanding window* ordenado por `AppointmentDay`: ancla al **tiempo de decisión** (la programación), no al orden de observación. Una cita programada hoy con fecha dentro de un mes puede legítimamente conocer el resultado de otra cita del mismo paciente programada después pero ejecutada antes.

**Auto-no-fuga (filas anómalas):** en las pocas filas con `AppointmentDay < ScheduledDay`, la propia fila figuraría entre las "previas" según la regla literal. La excluimos explícitamente — un outcome no puede ser su propio antecedente.

Salidas:
- `prior_appointment_count` — número de citas previas conocidas en `S`.
- `prior_noshow_rate` — tasa de no-asistencia en ese historial; **NaN para primeras visitas** (no es lo mismo que cero — refleja desconocimiento).
- `is_first_visit` — 1 si no hay historial previo.

> **Caveat — censura por la izquierda.** Dado que la base de datos cubre únicamente una ventana temporal limitada, estas tres variables **no** representan un historial clínico completo del paciente. `is_first_visit = 1` debe entenderse como *"primera visita observada"* o *"ausencia de historial previo observado"*, **no** como garantía de que el paciente nunca acudió antes a la clínica. Análogamente, `prior_appointment_count` y `prior_noshow_rate` resumen únicamente el historial visible en el periodo del dataset. Esta limitación —conocida como **censura por la izquierda**— se citará textualmente en la sección de limitaciones del TFG escrito.

In [7]:
df = prep.crear_features_historial_paciente(df)

print()
print("prior_appointment_count — describe:")
print(df['prior_appointment_count'].describe())
print()
print(f"prior_noshow_rate — describe (excluyendo {df['prior_noshow_rate'].isna().sum()} NaN de primeras visitas):")
print(df['prior_noshow_rate'].dropna().describe())
print()
print("is_first_visit:", df['is_first_visit'].value_counts().to_dict())

Historial paciente: 77024 primeras visitas (72.0%); prior_noshow_rate es NaN para esas filas (correcto, no son cero)



prior_appointment_count — describe:
count    106987.000000
mean          0.909568
std           3.701076
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          83.000000
Name: prior_appointment_count, dtype: float64

prior_noshow_rate — describe (excluyendo 77024 NaN de primeras visitas):
count    29963.000000
mean         0.198451
std          0.346393
min          0.000000
25%          0.000000
50%          0.000000
75%          0.333333
max          1.000000
Name: prior_noshow_rate, dtype: float64

is_first_visit: {1: 77024, 0: 29963}


## 7. División temporal por percentil 80 de `AppointmentDay`

Train: primeros 80% de fechas (`AppointmentDay <= corte`). Test: últimos 20% (`AppointmentDay > corte`). El corte es una fecha real del dataset (`interpolation="lower"`) y queda **incluida** en train.

Por qué temporal y no aleatoria: los pacientes aparecen varias veces en el dataset; un split aleatorio mezclaría citas del mismo paciente entre train y test, inflando artificialmente el rendimiento aparente del modelo predictivo.

El análisis causal IPW (Notebook 04) usa el **dataset completo** (`full_clean_v1.csv`), no el split de entrenamiento.

In [8]:
train, test, fecha_corte = prep.dividir_temporal(df, percentil=0.80)
print()
print(f"Fecha de corte: {fecha_corte.date()} (incluida en train)")
print(f"Train: {len(train):>7,} filas — {train['AppointmentDay'].min().date()} → {train['AppointmentDay'].max().date()}")
print(f"Test:  {len(test):>7,} filas — {test['AppointmentDay'].min().date()} → {test['AppointmentDay'].max().date()}")

División temporal en 2016-06-01 (P80): train=85657 (80.1%), test=21330 (19.9%)



Fecha de corte: 2016-06-01 (incluida en train)
Train:  85,657 filas — 2016-04-29 → 2016-06-01
Test:   21,330 filas — 2016-06-02 → 2016-06-08


## 8. Codificación por frecuencia de `Neighbourhood`

Ajuste **sólo** sobre el conjunto de entrenamiento (la frecuencia poblacional aprendida desde train se aplica a test y al dataset completo). NUNCA usamos `Showed_up` para codificar barrios — eso filtraría el outcome al modelo de propensity y contaminaría la interpretación causal.

La misma codificación (frecuencia de train) se aplica al `full_clean` que alimentará el IPW: así el barrio tiene el mismo significado numérico en ambos modelos (XGBoost predictivo y propensity score).

In [9]:
mapping_barrio = prep.ajustar_codificacion_frecuencia_barrio(train)
df = prep.aplicar_codificacion_frecuencia_barrio(df, mapping_barrio)
train = prep.aplicar_codificacion_frecuencia_barrio(train, mapping_barrio)
test = prep.aplicar_codificacion_frecuencia_barrio(test, mapping_barrio)

print()
print("neighbourhood_encoded — describe en full_clean:")
print(df['neighbourhood_encoded'].describe())

Codificación por frecuencia ajustada en train: 81 barrios únicos



neighbourhood_encoded — describe en full_clean:
count    106987.000000
mean          0.024751
std           0.017350
min           0.000012
25%           0.012492
50%           0.021388
75%           0.030832
max           0.070420
Name: neighbourhood_encoded, dtype: float64


## 9. Persistencia y verificación final

Guardamos tres ficheros versionados (`v1` = primera versión; cualquier cambio en covariables o regla de split genera `v2`):

- `full_clean_v1.csv` — dataset completo con todas las variables; entrada del Notebook 04 (IPW).
- `train_v1.csv`, `test_v1.csv` — split temporal para el modelo predictivo (Notebook 03).

Adicionalmente, persistimos un sidecar JSON con los números metodológicos críticos (fecha de corte, recuentos, % primeras visitas, anomalías documentadas) en `outputs/reportes/nb01_metadatos_v1.json`. Esto permite reproducir la narrativa de EDA y la sección de metodología del TFG escrito incluso si el notebook se vuelve a ejecutar meses después con un entorno ligeramente distinto.

In [10]:
# Guardado de datasets
df.to_csv(CSV_FULL_CLEAN, index=False)
train.to_csv(CSV_TRAIN, index=False)
test.to_csv(CSV_TEST, index=False)

print("Ficheros guardados en datos/procesados/:")
for ruta in (CSV_FULL_CLEAN, CSV_TRAIN, CSV_TEST):
    size_mb = ruta.stat().st_size / (1024 * 1024)
    print(f"  {ruta.name:<22} {size_mb:6.2f} MB")

# Sidecar de metadatos
metadatos = prep.escribir_metadatos_nb01(
    REPORTES / "nb01_metadatos_v1.json",
    seed=SEED,
    df_bruto=df_bruto,
    df_full=df,
    df_train=train,
    df_test=test,
    fecha_corte=fecha_corte,
    mapping_barrio=mapping_barrio,
)
print()
print("Metadatos NB01:")
for k, v in metadatos.items():
    print(f"  {k}: {v}")

Ficheros guardados en datos/procesados/:
  full_clean_v1.csv       15.84 MB
  train_v1.csv            12.66 MB
  test_v1.csv              3.17 MB
Metadatos NB01 guardados en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/reportes/nb01_metadatos_v1.json



Metadatos NB01:
  generado_en: 2026-05-10T23:18:53.652425+00:00
  seed: 42
  n_filas_brutas: 106987
  n_filas_total: 106987
  n_filas_train: 85657
  n_filas_test: 21330
  fecha_corte_appointmentday: 2016-06-01
  pct_atendieron: 0.7973585575817622
  n_primeras_visitas: 77024
  pct_primeras_visitas: 0.7199379363847944
  n_barrios_train: 81
  n_barrios_test_oov: 0
  n_anomalia_appt_lt_sched: 5
  media_prior_noshow_rate: 0.19845073799172627
  n_edad_negativa_eliminadas: 0
  valores_unicos_handcap: [False, True]
  scheduled_hour_eliminada_por_constante: True


In [11]:
# Verificación final: invariantes que un notebook NB02–06 puede asumir
# como ciertas. Cualquier ruptura aquí debe impedir el guardado a producción.
COLUMNAS_ESPERADAS = {
    # Originales preservadas (PatientId pasa a int64)
    "PatientId", "AppointmentID", "Gender", "ScheduledDay", "AppointmentDay",
    "Age", "Neighbourhood", "Scholarship", "Hipertension", "Diabetes",
    "Alcoholism", "Handcap", "SMS_received", "Showed_up", "Date.diff",
    # Derivadas
    "lead_time", "lead_time_bin", "age_band",
    "comorbidity_count", "chronic_flag",
    "scheduled_weekday", "scheduled_month",
    "appointment_weekday", "appointment_month",
    # Historial paciente (regla estricta)
    "prior_appointment_count", "prior_noshow_rate", "is_first_visit",
    # Codificación
    "neighbourhood_encoded",
}

faltantes = COLUMNAS_ESPERADAS - set(df.columns)
extras = set(df.columns) - COLUMNAS_ESPERADAS
assert not faltantes, f"Faltan columnas esperadas: {faltantes}"

# Tipos y dominios de las variables clave
assert df['Showed_up'].dtype == 'int64', f"Showed_up debe ser int64, es {df['Showed_up'].dtype}"
assert df['Showed_up'].isin([0, 1]).all(), "Showed_up tiene valores fuera de {0, 1}"
assert df['PatientId'].dtype == 'int64', f"PatientId debe ser int64, es {df['PatientId'].dtype}"
assert df['neighbourhood_encoded'].notna().all(), "neighbourhood_encoded tiene NaN"
assert (df['prior_appointment_count'] >= 0).all(), "prior_appointment_count negativo"

# Invariante de la regla de fuga: prior_noshow_rate es NaN exactamente para
# primeras visitas (donde no hay historial previo).
mask_first = df['is_first_visit'] == 1
assert df.loc[mask_first, 'prior_noshow_rate'].isna().all(),     "Hay primeras visitas con prior_noshow_rate no-NaN — fuga sospechosa"
assert df.loc[~mask_first, 'prior_noshow_rate'].notna().all(),     "Hay no-primeras visitas con prior_noshow_rate NaN — inconsistencia"

# Totalidad de la división temporal: train + test debe sumar exactamente
# las filas del dataset completo.
assert len(df) == len(train) + len(test),     f"Pérdida en split: full={len(df)}, train+test={len(train)+len(test)}"

# Disjuntez temporal: la última cita de train debe ser <= la primera de test.
assert train['AppointmentDay'].max() <= test['AppointmentDay'].min(),     "Solapamiento temporal entre train y test"

# Codificación de barrios: aplicada en todas las filas (sin NaN, ya verificado)
# y consistente con el mapping ajustado en train.
n_barrios_train = len(set(train['Neighbourhood']))
n_barrios_test_oov = len(set(test['Neighbourhood']) - set(mapping_barrio.keys()))
assert n_barrios_train == len(mapping_barrio),     "El mapping de barrios no coincide con los barrios únicos de train"

print("Todas las comprobaciones finales pasaron.")
print(f"  Columnas esperadas: {len(COLUMNAS_ESPERADAS)}, presentes en full_clean: {len(set(df.columns) & COLUMNAS_ESPERADAS)}")
print(f"  Columnas extra (informativas): {sorted(extras) if extras else 'ninguna'}")
print(f"  Forma final de full_clean: {df.shape}")
print(f"  Barrios únicos en train: {n_barrios_train}; en test fuera de train: {n_barrios_test_oov}")

Todas las comprobaciones finales pasaron.
  Columnas esperadas: 28, presentes en full_clean: 28
  Columnas extra (informativas): ninguna
  Forma final de full_clean: (106987, 28)
  Barrios únicos en train: 81; en test fuera de train: 0
